# Assignment Overview

In this assignment I will work with Electrocardiogram (from now on ECG) signlas. They are used to monitor health activity. I will be further expandig the ecg.ipynb file by choosing an additional machine learning model of my choice and training it using the ECG signals dataset and then evaluating the performance of the model. Furthermore i will introduce a new metric, which will help me compare the models performance to other existing models.

The following code cannot be run on its own, as it is a continuation of the ecg.ipynb file. So if possible just concatenate the code before computing the confusion matrix and after calculating the three models.

# Import Libraries

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns

from sklearn.metrics import (
    f1_score,
)
from sklearn.manifold import TSNE
from sklearn.svm import NuSVC

# `Selecting a Machine Learning Model`

Before choosing a model i have seen that all the model chosen (Logistic Regression, ExtraTreeClassifier, and LGBMClassifier) are all algorithms used for classification. Which is why i will also use a classification model. My first thaught is to use the Support Vector Machine classifier. Reason being it is effective on medical data and can handle inbalanced data. While reading through some article i stumbled upon an article about 'Everything About Support Vector Classification - Above and Beyond (from Ashwin Raj)' [Link to article](https://towardsdatascience.com/everything-about-svm-classification-above-and-beyond-cc665bfd993e) there i saw the NuSVC() model which is also used for classification purposes. I read upon it a bit and found out that SVC and NuSVC are mathematically equivalent with both method based on the library [LIBSVM](https://www.csie.ntu.edu.tw/~cjlin/libsvm/). The main differnce is the use case of the parameter 'c' for SVC and 'nu' for NuSVC. Here is the user guide for the parameter 'c' for SVC and 'nu' for NuSVC [Link to user guide](https://scikit-learn.org/stable/modules/svm.html#nu-svc). 

Some general characteristics of the NuSVC model are:

- less sensitive to outliers than the SVC model, beneficial for noisy datasets.
- parameter 'nu', which is an upperbound on the fraction of margin errors and a lowerbound on the fraction of support vectors. With that i will get a good tradeoff between bias and variance.
- supports multi-class classification.
- Kernel function to capture complex relationships in data.
- efficient in terms of memory usage and computation time, especially when dealing with large datasets such as ECG signal data.
- less interpretable.

With that in mind i explored the data and visualized it to understand its characteristics.

- I looked for class separability: Are the classes well-separated, or do they overlap significantly?
- I explored feature distributions and tried to understand their relationship with the target variable.
- And asses the data balance.

In [ ]:
class_distribution = features_df_trainvalid["diagnosis"].value_counts().reset_index()
class_distribution.columns = ["Diagnosis", "Count"]

fig = px.bar(class_distribution, x="Diagnosis", y="Count", title="Class Distribution")
fig.show()

By plotting the distribution of the target variable (diagnosis). We can see the classes are imbalanced with more myocardial infarction.

For further inspection i chose to visualize the data distribution and the seperability of my data with t-SNE (t-Distributed Stochastic Neighbor Embedding).

In [ ]:
tsne = TSNE(n_components=2, random_state=42)
tsne_result = tsne.fit_transform(features_df_trainvalid[feature_columns])

plt.figure(figsize=(10, 8))
sns.scatterplot(x=tsne_result[:, 0], y=tsne_result[:, 1], hue=features_df_trainvalid["diagnosis"])
plt.title("t-SNE Visualization of Data")
plt.show()

I reduced the dimensionality of my data to 2D and plotted the data points. The color of the data points represent the diagnosis class. With that i can roughly differentiate the classes. One can see some degree of seperation between the two classes (Healthy control and Bundle branch block). But since the class is imbalanced and the data points of different color overlap indicates, that the classes are not well seperated. So NuSVM may not be the best choice! But i still want to compare how it performs and how it compares tho the other models.

# `Training and Evaluation`

In [ ]:
best_nusvm_model = NuSVC(kernel='linear', nu=0.01, random_state=42)

nusvm_results = evaluate_folds(
    Xy=features_df_trainvalid,
    feature_columns=feature_columns,
    label_column=label_column,
    model=best_nusvm_model,
    metrics=metrics,
    class_weights=True,
)
nusvm_results["model"] = "nusvm"

results = pd.concat(
    [
        lr_results,
        et_results,
        lgbm_results,
        nusvm_results,
    ]
)

scores = results[results["metric"] != "confusion_matrix"]

px.scatter(scores, facet_col="metric", facet_row="split", x="model", y="value", color="fold")


In [ ]:
scores = results[results["metric"] != "confusion_matrix"]

g = sns.catplot(
    x="metric",
    y="value",
    hue="model",
    col="split",
    data=scores,
    kind="bar",
    height=5,
    aspect=1.2,
)

g.set_axis_labels("Metric", "Metric Value")
g.set_titles("{col_name} Dataset")

for ax in g.axes.flat:
    for p in ax.patches:
        ax.annotate(
            f"{p.get_height():.2f}",
            (p.get_x() + p.get_width() / 2, p.get_height()),
            ha="center", va="bottom", fontsize=10
        )

plt.show()


A short description of the metrics used:

1) Accuracy: This metric measures the overall correctness of the model's predictions.
2) Balanced Accuracy: It is similar to accuracy but takes into account class imbalances
3) Matthews Correlation Coefficient (from now on MCC): MCC is a measure of the quality of binary classifications, considering both true positives and true negatives.

Interpretation of the metrics in the training dataset:

1) Accuracy: All models have a high accuracy and they are performing well on the training dataset.
2) Balanced Accuracy: An excellent performance on the trainig set for all models.
3) MCC: NuSVC is very close to 1, indicating strong performance.

Interpretation of the metrics in the validation dataset:

1) Accuracy: Logistic Regression has the lowest accuracy of 54% and LGBMClassifier performed the best on the validation dataset with 73% accuracy.
2) Balanced Accuracy: They are overall relatively low, indicating that the model might not generalize well to the validation set.
3) MCC: NuSVC achieved a score of 19%, indicating better than random prediction but room for improvement.

Summary:

While all the models performed very good on the training dataset, all close to 1, their performance on validation set drops. this was expected as the model may not generalize as well to unseen data. LGBMClassifier seems to generalize the best to the validation set, with the highest MCC and accuracy. ExtraTreeClassifier generalizes the worst on the validation set.

My chosen model (NuSVC) performs reasonably well but as expected it does not generalize well to the validation set. This is due to the imbalanced dataset and the overlapping data points. The model is not able to differentiate the classes well. This is also reflected in the MCC score, which is very low. 

# `Additional Metrics and Comparison`

In [ ]:
metrics = {
    "confusion_matrix": confusion_matrix,
    "accuracy": accuracy_score,
    "balanced_accuracy": balanced_accuracy_score, 
    "matthews_corrcoef": matthews_corrcoef, 
    "f1_score_weighted": lambda y_true, y_pred: f1_score(y_true, y_pred, average='weighted')
}

I chose to use the weighted F1 score as additional metrics. The weighted F1 score is the harmonic mean of precision (minimizing false positives) and recall (capturing all positive cases). It is a good metric to use because the dataset of ECG classification is imbalanced therfore only accuracy can be misleading and the weighted F1 score considers both false positives and false negatives, making it more robust. The weighted F1 score gives also an insight to to performance for each class. This helps identifying which class the model struggles with the most and which are valuable. I could also have chosen the two metrics precision and recall individually, but i wanted a single metric that considers the tradeoff of both. It is also easy to interpret the results of the weighted F1 score, as it is a value between 0 and 1, where 1 is the best possible score, indicating best model performance. [Link to article](https://towardsdatascience.com/micro-macro-weighted-averages-of-f1-score-clearly-explained-b603420b292f#33e1)

Remark: I will also repeat the evaluation on the new metric with my chosen model (NuSVC).

In [ ]:
lr_model = LogisticRegression(solver="liblinear", max_iter=1000) 
#lr_model = LogisticRegression()
lr_results = evaluate_folds(
    Xy=features_df_trainvalid,
    feature_columns=feature_columns,
    label_column=label_column,
    model=lr_model,
    metrics=metrics,
    class_weights=True,
)
lr_results["model"] = "lr"

et_model = ExtraTreeClassifier()
et_results = evaluate_folds(
    Xy=features_df_trainvalid,
    feature_columns=feature_columns,
    label_column=label_column,
    model=et_model,
    metrics=metrics,
    class_weights=True,
)
et_results["model"] = "et"

lgbm_model = LGBMClassifier()
lgbm_results = evaluate_folds(
    Xy=features_df_trainvalid,
    feature_columns=feature_columns,
    label_column=label_column,
    model=lgbm_model,
    metrics=metrics,
    class_weights=True,
)
lgbm_results["model"] = "lightgbm"

best_nusvm_model = NuSVC(kernel='linear', nu=0.01, random_state=42)
nusvm_results = evaluate_folds(
    Xy=features_df_trainvalid,
    feature_columns=feature_columns,
    label_column=label_column,
    model=best_nusvm_model,
    metrics=metrics,
    class_weights=True,
)
nusvm_results["model"] = "nusvm"

results = pd.concat(
    [
        lr_results,
        et_results,
        lgbm_results,
        nusvm_results,
    ]
)

scores = results[results["metric"] != "confusion_matrix"]
px.scatter(scores, facet_col="metric", facet_row="split", x="model", y="value", color="fold")

In [ ]:
scores = results[results["metric"] != "confusion_matrix"]

g = sns.catplot(
    x="metric",
    y="value",
    hue="model",
    col="split",
    data=scores,
    kind="bar",
    height=5,
    aspect=1.2,
)

g.set_axis_labels("Metric", "Metric Value")
g.set_titles("{col_name} Dataset")

for ax in g.axes.flat:
    for p in ax.patches:
        ax.annotate(
            f"{p.get_height():.2f}", 
            (p.get_x() + p.get_width() / 2, p.get_height()),
            ha="center", va="bottom", fontsize=10
        )

plt.show()

All models performed well on trainig set and showed with that some signs of potential overfitting. Regarding the validation set one can see that LGBMClassifier performed the best among the models and right behind that the NuSVC model. They both have high accuracy and F1 score, indicating good classification. However the balanced accuracy is relatively low, indicating that the model might not generalize well to the validation set. The MCC score is also relatively low, indicating better than random prediction but room for improvement.